## Segment spontaneous data

Segments spontaneous data and saves thresholds and onsets for all fish.

Import libraries:

In [ ]:
# --- Imports and path setup (paths from code/paths.py) ---
import numpy as np
import matplotlib.pyplot as plt
def inline():
    %matplotlib inline
    plt.close('all')
def widget():
    %matplotlib widget
    plt.close('all')
import sys
from pathlib import Path
_code = Path.cwd() if (Path.cwd() / "paths.py").exists() else Path.cwd().parent
if str(_code.resolve()) not in [str(Path(p).resolve()) for p in sys.path]:
    sys.path.insert(0, str(_code.resolve()))
from paths import SPEED_SEGMENTED_DIR, media_dir
path = str(SPEED_SEGMENTED_DIR)
path_media = str(media_dir(path))

%load_ext autoreload
%autoreload 2
from functions.visual_flow_behavior_plotting import *
from functions.visual_flow_behavior_analysis import *
from functions.brain_analysis import *
from functions.utilities import *
from functions.feedback_models import *
from functions.freely_swimming import *

import os
import h5py
from ipywidgets import interact, IntSlider
from scipy import fft
from scipy.ndimage import gaussian_filter
from skimage import io
import pickle
from sklearn.preprocessing import scale
from mpl_toolkits.axes_grid1.axes_divider import make_axes_locatable
from scipy.stats import iqr
import pandas as pd

In [ ]:
# path, path_media set in first cell

In [ ]:
# path_media created in first cell

#### Import data:

In [ ]:
fish_ids_young=['13_06May24_1','13_30Oct24_3','13_30Oct24_4','14_27Mar24_1','14_31Oct24_1']

In [ ]:
fish_ids_intermediate=['17_21May24_2','18_22May24_1','18_22May24_2','19_23May24_5','20_05Jun24_1','20_05Jun24_2','20_12Jun24_1','21_06Dec24_1','21_06Dec24_2','21_06Jun24_1','25_10Jun24_1']

In [ ]:
fish_ids_old=['26_02May24_2','27_12Jun24_1','28_04May24_1','30_06May24_2','34_11Dec24_1','34_11Dec24_3']

In [ ]:
fish_ids=fish_ids_young+fish_ids_intermediate+fish_ids_old

In [ ]:
fish_ids

In [ ]:
len(fish_ids)

Load:

In [ ]:
speed_clean = pd.read_hdf(os.path.join(path, f"speed_clean.h5"), key='df')

In [ ]:
speed_clean

In [ ]:
t_c_list=[]
vfish_c_list=[]
vtail_c_list=[]
mean_thrust_c_list=[]
angletail_c_list=[]

for fish_ID in tqdm(fish_ids):
    speed_clean_fish=subset_by_fish(fish_ID,DF1=speed_clean)[0]
    t_c_list.append(speed_clean_fish['t_c'].values)
    vfish_c_list.append(speed_clean_fish['vfish_c'].values)
    vtail_c_list.append(speed_clean_fish['vtail_c'].values)    
    mean_thrust_c_list.append(speed_clean_fish['mean_thrust_c'].values)
    angletail_c_list.append(speed_clean_fish['angletail_c'].values)

In [ ]:
thrust_thr_list=[]
for i,mean_thrust_c in enumerate(mean_thrust_c_list):

    thrust_thr_list.append(10**threshold_li(np.log10(mean_thrust_c[~np.isnan(mean_thrust_c)])))

In [ ]:
@interact(i=(0,len(mean_thrust_c_list)-1,1))
def plot(i=0):
    default()
    widget()
    fig,ax=plt.subplots(figsize=(6.4,4.8),dpi=150)
    ax.hist(np.log10(mean_thrust_c_list[i]),bins=100,histtype='step',density=True)
    ax.axvline(np.log10(thrust_thr_list[i]),c='k')
    plt.show()

In [ ]:
mask_movetail_list=[]
for i,mean_thrust_c in tqdm(enumerate(mean_thrust_c_list)):
    mask_movetail_list.append(compute_mask_thrust_thr(mean_thrust_c,mean_thrust_thr=thrust_thr_list[i],pad_time=0.06/2))

In [ ]:
i=13
dt=1/300
default()
inline()
fig,ax=plt.subplots(figsize=(6.4,4.8),dpi=150)
ax.plot(t_c_list[i],vtail_c_list[i]/1000,'-')
ax.plot(t_c_list[i],angletail_c_list[i]/10,'-')
ax.plot(t_c_list[i],mean_thrust_c_list[i],'-')
ax.axhline(thrust_thr_list[i],c='k')
ax.plot(t_c_list[i][mask_movetail_list[i]], np.zeros(np.sum(mask_movetail_list[i])),'.')

plt.show()

In [ ]:
onset_indices_list=[]
for i in tqdm(range(len(mean_thrust_c_list))):
    onset_indices_list.append(get_onsets_clean(mask_movetail_list[i],t_c_list[i],mean_thrust_c_list[i],mean_thrust_thr=thrust_thr_list[i]))

In [ ]:
initial_vfish_list=[]
initial_vtail_list=[]
bout_duration_list=[]
for i in tqdm(range(len(mean_thrust_c_list))):
    initial_vfish,initial_vtail,bout_duration=get_onset_stats(mask_movetail_list[i],t_c_list[i],mean_thrust_c_list[i],onset_indices_list[i],vfish_c_list[i],vtail_c_list[i])
    initial_vfish_list.append(initial_vfish)
    initial_vtail_list.append(initial_vtail)
    bout_duration_list.append(bout_duration)

In [ ]:
i=10
dt=1/300
t_before=0.1
t_after=1
@interact(n=(0,len(onset_indices_list[i])-1,1))
def plot(n=0):
    default()
    widget()
    fig,ax=plt.subplots(figsize=(6.4,4.8),dpi=150)
    ax.axhline(0,color='lightgray')
    ax.axvline(0,color='lightgray')
    t_onset=t_c_list[i][onset_indices_list[i][n]]
    indices=(t_c_list[i]>t_onset-t_before)&(t_c_list[i]<t_onset+t_after)
    ax.plot(t_c_list[i][indices]-t_onset,10*vfish_c_list[i][indices],'.-',alpha=0.6)
    ax.plot(t_c_list[i][indices]-t_onset,vtail_c_list[i][indices],'.-',alpha=0.6)
    ax.axvline(bout_duration_list[i][n],c='k')
    ax.set_xlim(-t_before,t_after)
    plt.show()

In [ ]:
i=10
dt=1/300
t_before=0.1
t_after=1
@interact(n=(0,len(onset_indices_list[i])-1,1))
def plot(n=0):
    default()
    widget()
    fig,ax=plt.subplots(figsize=(6.4,4.8),dpi=150)
    ax.axhline(0,color='lightgray')
    ax.axvline(0,color='lightgray')
    t_onset=t_c_list[i][onset_indices_list[i][n]]
    indices=(t_c_list[i]>t_onset-t_before)&(t_c_list[i]<t_onset+t_after)
    ax.plot(t_c_list[i][indices]-t_onset,vfish_c_list[i][indices]/1000,'.-',alpha=0.6)
    ax.plot(t_c_list[i][indices]-t_onset,vtail_c_list[i][indices]/10000,'.-',alpha=0.6)
    ax.plot(t_c_list[i][indices]-t_onset,mean_thrust_c_list[i][indices],'.-',alpha=0.6)
    ax.axvline(bout_duration_list[i][n],c='k')
    ax.axhline(thrust_thr_list[i],c='k')
    ax.set_xlim(-t_before,t_after)
    plt.show()

In [ ]:
i=1
default()
inline()
fig,ax=plt.subplots(figsize=(6.4,4.8),dpi=150)
ax.axhline(0,c='lightgray')
ax.axvline(0,c='lightgray')
ax.scatter(bout_duration_list[i],initial_vfish_list[i],s=20,alpha=0.3,edgecolor='none')

ax.set_xlabel('bout duration (s)')
ax.set_ylabel('initial fish speed (mm/s)')

plt.show()

In [ ]:
thrust_thresholds = pd.DataFrame()

for i,fish_ID in enumerate(fish_ids):
    
    thrust_thresholds_fish = pd.DataFrame({
                            'ID': fish_ID,
                            'thrust_thr': thrust_thr_list[i],
                            'mask_movetail': mask_movetail_list[i],
                            })
    
    thrust_thresholds=pd.concat([thrust_thresholds, thrust_thresholds_fish], ignore_index=True)

In [ ]:
onsets = pd.DataFrame()

for i,fish_ID in enumerate(fish_ids):
    
    onsets_fish = pd.DataFrame({
                            'ID': fish_ID,
                            'onset_indices': onset_indices_list[i],
                            'initial_vfish': initial_vfish_list[i],
                            'initial_vtail': initial_vtail_list[i],
                            'bout_duration': bout_duration_list[i],
                            })
    
    onsets=pd.concat([onsets, onsets_fish], ignore_index=True)

Save:

Load: